In [ ]:
import pandas as pd
import networkx as nx
import community as community_louvain
import os

DATA_DIR = "./data"
OUTPUT_DIR = "./outputs"

from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
pd.read_excel(os.path.join(OUTPUT_DIR, "dataset_with_sentiment.xlsx"))

print(df.shape)
df.head()

(5715, 26)


,tweet_id,display_name,username,followers,tweet,created_at,lang,user_location,tweet_location,likes,...,location_clean,city_clean,sentiment_roberta,confidence_roberta,polarity_roberta,sentiment_model2,confidence_model2,polarity_model2,sentiment,sentiment_score
0,1406970754688717056,Яizal do,afrkml,350301,Gejala yg ditimbulkan memang umum dan sudah di...,Mon Jun 21 13:42:22 +0000 2021,in,Dukung kami di sini →,NaN,245,...,dukung kami di sini,NaN,negative,0.752307,-0.752307,negative,0.995110,-0.995110,negative,-0.995110
1,1409518162362568960,🍒,claudiaars__,230,Until this minute masih ga habis pikir bisa2ny...,Mon Jun 28 14:24:52 +0000 2021,in,Jakarta,NaN,0,...,jakarta,jakarta,negative,0.996685,-0.996685,negative,0.619597,-0.619597,negative,-0.619597
2,1408483130994946048,R30_K03 🇮🇩,ryuuken86,24,Hari jum'at 24 juni. Guru SMP ku meninggal di ...,Fri Jun 25 17:52:01 +0000 2021,in,Indonesia,NaN,0,...,indonesia,NaN,negative,0.616726,-0.616726,positive,0.891653,0.891653,positive,0.891653
3,1409494375294390016,M Taufik Zoelkifli,emtezet,2127,Singapura sudah akan beranjak dari Pandemi men...,Mon Jun 28 12:50:20 +0000 2021,in,Jakarta,"Cempaka Putih, Indonesia",2,...,cempaka putih indonesia,NaN,positive,0.778616,0.778616,positive,0.954192,0.954192,positive,0.954192
4,1407938212631237120,rahardjoguyub,guyubrahardjo,598,@CNNIndonesia @PutraWadapi Kondisi jakarta yg ...,Thu Jun 24 05:46:42 +0000 2021,in,"DKI Jakarta, Indonesia",NaN,0,...,dki jakarta indonesia,jakarta,neutral,0.654501,0.000000,neutral,0.889500,0.000000,neutral,0.000000


In [ ]:
top_complete = pd.read_excel(os.path.join(OUTPUT_DIR, "complete_network_centrality.xlsx"))

top_complete.head()

,username,pagerank,in_degree,out_degree,betweenness
0,afrkml,0.003688,4,1,0.0
1,tirta_cipeng,0.003668,3,1,0.0
2,dinkesjateng,0.003091,3,1,0.0
3,kegblgnunfaedh,0.002749,16,0,0.0
4,convomf,0.002663,16,0,0.0


In [ ]:
edges_weighted = pd.read_excel(os.path.join(OUTPUT_DIR, "weighted_edges.xlsx"))

edges_weighted.head()

,source,target,sentiment,weight
0,0bungkal,polres_ponorogo,neutral,1
1,0jenry1,PNS_Ababil,neutral,1
2,1404stj,1404stj,positive,2
3,1404stj,UNICEF,positive,1
4,1404stj,jokowi,positive,1


--------------------- COMPLETE NETWORK ------------------------------

Build Complete Network

In [187]:
G = nx.from_pandas_edgelist(
    edges_weighted,
    source="source",
    target="target",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

print(
    "Nodes:",
    G.number_of_nodes()
)

print(
    "Edges:",
    G.number_of_edges()
)

Nodes: 2166
Edges: 1767


Louvain Community Detection

In [188]:
partition = community_louvain.best_partition(
    G.to_undirected(),
    random_state=42
)

In [189]:
#Check number of community
print(
    "Number of communities:",
    len(
        set(partition.values())
    )
)

Number of communities: 861


Community DataFrame

In [190]:
community_df = pd.DataFrame(
    partition.items(),
    columns=[
        "username",
        "community"
    ]
)

community_df.head()

,username,community
0,0bungkal,0
1,polres_ponorogo,0
2,0jenry1,1
3,PNS_Ababil,1
4,1404stj,28


In [ ]:
community_df.to_excel(
    os.path.join(OUTPUT_DIR, "community_assignment.xlsx"),
    index=False
)

Community Size

In [192]:
#Jumlah member tiap komunitas
community_size = (
    community_df["community"]
    .value_counts()
    .reset_index()
)

community_size.columns = [
    "community",
    "members"
]

community_size

,community,members
0,37,85
1,28,60
2,80,46
3,78,35
4,253,28
...,...,...
856,848,1
857,849,1
858,850,1
859,859,1


In [193]:
community_size = (
    community_size
    .sort_values(
        "members",
        ascending=False
    )
)

community_size.head(20)

,community,members
0,37,85
1,28,60
2,80,46
3,78,35
4,253,28
5,685,21
6,49,18
7,35,17
8,64,16
9,134,15


In [ ]:
community_size.to_excel(
    os.path.join(OUTPUT_DIR, "community_size.xlsx"),
    index=False
)

Top Influencer per Community

In [195]:
#Combine Community with PageRank
community_centrality = (
    community_df
    .merge(
        top_complete,
        on="username",
        how="left"
    )
)

community_centrality.head()

,username,community,pagerank,in_degree,out_degree,betweenness
0,0bungkal,0,0.000206,0,1,0.0
1,polres_ponorogo,0,0.000381,1,0,0.0
2,0jenry1,1,0.000206,0,1,0.0
3,PNS_Ababil,1,0.000381,1,0,0.0
4,1404stj,28,0.000359,1,3,0.0


In [196]:
#Find Influencer
top_influencer_community = (
    community_centrality
    .sort_values(
        "pagerank",
        ascending=False
    )
    .groupby("community")
    .first()
    .reset_index()
)

top_influencer_community

,community,username,pagerank,in_degree,out_degree,betweenness
0,0,polres_ponorogo,0.000381,1,0,0.0
1,1,PNS_Ababil,0.000381,1,0,0.0
2,2,wmmetawin,0.000381,1,0,0.0
3,3,nuellubis,0.000381,1,0,0.0
4,4,nmonarizqa,0.000381,1,0,0.0
...,...,...,...,...,...,...
856,856,CakCaping,0.000226,1,0,0.0
857,857,Kamaestungkara,0.000294,1,0,0.0
858,858,FWBeks,0.000294,1,0,0.0
859,859,youthssfm,0.001360,1,1,0.0


In [ ]:
top_influencer_community.to_excel(
    os.path.join(OUTPUT_DIR, "top_influencer_per_community.xlsx"),
    index=False
)

Dominant Sentiment per Community

In [198]:
#Combined tweet with community
community_tweets = (
    df.merge(
        community_df,
        on="username",
        how="inner"
    )
)

community_tweets.head()

,tweet_id,display_name,username,followers,tweet,created_at,lang,user_location,tweet_location,likes,...,city_clean,sentiment_roberta,confidence_roberta,polarity_roberta,sentiment_model2,confidence_model2,polarity_model2,sentiment,sentiment_score,community
0,1406970754688717056,Яizal do,afrkml,350301,Gejala yg ditimbulkan memang umum dan sudah di...,Mon Jun 21 13:42:22 +0000 2021,in,Dukung kami di sini →,NaN,245,...,NaN,negative,0.752307,-0.752307,negative,0.995110,-0.995110,negative,-0.995110,37
1,1407938212631237120,rahardjoguyub,guyubrahardjo,598,@CNNIndonesia @PutraWadapi Kondisi jakarta yg ...,Thu Jun 24 05:46:42 +0000 2021,in,"DKI Jakarta, Indonesia",NaN,0,...,jakarta,neutral,0.654501,0.000000,neutral,0.889500,0.000000,neutral,0.000000,80
2,1407197145682247936,tikatiki,tikaresianaa,13,@milasantika06 Nah betul tuh\nTapi akhir akhir...,Tue Jun 22 04:41:58 +0000 2021,in,"Purwokerto Utara, Indonesia",NaN,0,...,NaN,negative,0.996815,-0.996815,negative,0.995957,-0.995957,negative,-0.995957,798
3,1404754951893443072,pin.,albusvenetus,459,"@bertanyarl 3,5,8,9,10 ini yg blm\n1. Taun lal...",Tue Jun 15 10:57:34 +0000 2021,in,She/her 20+,NaN,0,...,NaN,negative,0.736742,-0.736742,negative,0.984341,-0.984341,negative,-0.984341,14
4,1406566917880434944,Askrlfess,askrlfess,707097,[askrl] penyakit dbd sama malaria sama gak sih?,Sun Jun 20 10:57:40 +0000 2021,in,"Jakarta, Indonesia",NaN,0,...,jakarta,negative,0.719211,-0.719211,negative,0.956276,-0.956276,negative,-0.956276,85


In [199]:
#Count sentiment distributoin
community_sentiment = (
    community_tweets
    .groupby(
        [
            "community",
            "sentiment"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

community_sentiment.head()

,community,sentiment,count
0,0,neutral,1
1,1,neutral,1
2,2,negative,1
3,3,neutral,1
4,4,neutral,1


In [200]:
#find dominant sentiment
dominant_sentiment = (
    community_sentiment
    .sort_values(
        "count",
        ascending=False
    )
    .groupby("community")
    .first()
    .reset_index()
)

dominant_sentiment

,community,sentiment,count
0,0,neutral,1
1,1,neutral,1
2,2,negative,1
3,3,neutral,1
4,4,neutral,1
...,...,...,...
856,856,neutral,1
857,857,negative,1
858,858,negative,1
859,859,neutral,1


In [ ]:
dominant_sentiment.to_excel(
    os.path.join(OUTPUT_DIR, "dominant_sentiment_per_community.xlsx"),
    index=False
)

Dominant Keywords per Community

In [202]:
def get_top_keywords(
    texts,
    n=10
):

    vectorizer = CountVectorizer(
        max_features=1000
    )

    X = vectorizer.fit_transform(
        texts.dropna()
    )

    words = (
        vectorizer
        .get_feature_names_out()
    )

    freq = X.sum(
        axis=0
    ).A1

    keyword_df = pd.DataFrame({
        "keyword": words,
        "frequency": freq
    })

    return (
        keyword_df
        .sort_values(
            "frequency",
            ascending=False
        )
        .head(n)
    )

In [203]:
#loop for each community
keyword_results = []

for community_id in sorted(
    community_tweets[
        "community"
    ].unique()
):

    texts = (
        community_tweets[
            community_tweets[
                "community"
            ]
            == community_id
        ]["tweet_clean"]
    )

    top_keywords = (
        get_top_keywords(
            texts,
            n=10
        )
    )

    top_keywords[
        "community"
    ] = community_id

    keyword_results.append(
        top_keywords
    )

In [204]:
community_keywords = pd.concat(
    keyword_results,
    ignore_index=True
)

community_keywords.head()

,keyword,frequency,community
0,bungkal,3,0
1,puskesmas,2,0
2,anwar,1,0
3,berdarah,1,0
4,bersama,1,0


In [ ]:
community_keywords.to_excel(
    os.path.join(OUTPUT_DIR, "community_keywords.xlsx"),
    index=False
)

----------------- POSITIVE NETWORK ------------------------------

In [206]:
positive_edges = (
    edges_weighted[
        edges_weighted["sentiment"]
        == "positive"
    ]
)

In [207]:
print(
    "Positive edges:",
    len(positive_edges)
)

Positive edges: 318


In [208]:
G_positive = nx.from_pandas_edgelist(
    positive_edges,
    source="source",
    target="target",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

In [209]:
print(
    "Nodes:",
    G_positive.number_of_nodes()
)

print(
    "Edges:",
    G_positive.number_of_edges()
)

Nodes: 378
Edges: 318


Community Detection

In [210]:
partition_positive = (
    community_louvain.best_partition(
        G_positive.to_undirected(),
        random_state=42
    )
)

print(
    "Number of communities:",
    len(
        set(
            partition_positive.values()
        )
    )
)

Number of communities: 200


In [211]:
community_positive_df = pd.DataFrame(
    partition_positive.items(),
    columns=[
        "username",
        "community"
    ]
)

community_positive_df.head()

,username,community
0,1404stj,1
1,UNICEF,1
2,jokowi,1
3,93dancan,2
4,vincentrcrd,2


In [ ]:
community_positive_df.to_excel(
    os.path.join(OUTPUT_DIR, "positive_community_assignment.xlsx"),
    index=False
)

Community Size

In [213]:
community_size_positive = (
    community_positive_df["community"]
    .value_counts()
    .reset_index()
)

community_size_positive.columns = [
    "community",
    "members"
]

community_size_positive = (
    community_size_positive
    .sort_values(
        "members",
        ascending=False
    )
)

community_size_positive.head(20)

,community,members
0,1,17
1,12,14
2,130,11
3,150,10
4,133,9
5,72,7
6,64,6
7,128,6
8,37,5
9,38,5


In [ ]:
community_size_positive.to_excel(
    os.path.join(OUTPUT_DIR, "positive_community_size.xlsx"),
    index=False
)

Top Influencer Community

In [ ]:
top_positive = pd.read_excel(os.path.join(OUTPUT_DIR, "positive_network_centrality.xlsx"))
top_positive.head()

,username,pagerank,in_degree,out_degree,betweenness
0,anne_khanza,0.005618,1,1,0.0
1,anotherorion,0.005618,1,1,0.0
2,sitimustiani,0.005618,1,1,0.0
3,sksui,0.005618,1,1,0.0
4,amaninanaeeim,0.005618,1,1,0.0


In [216]:
community_centrality_positive = (
    community_positive_df
    .merge(
        top_positive,
        on="username",
        how="left"
    )
)

community_centrality_positive.head()

,username,community,pagerank,in_degree,out_degree,betweenness
0,1404stj,1,0.001468,1,3,0.0
1,UNICEF,1,0.001156,1,0,0.0
2,jokowi,1,0.001395,2,0,0.0
3,93dancan,2,0.000844,0,1,0.0
4,vincentrcrd,2,0.001561,1,0,0.0


In [217]:
top_influencer_positive = (
    community_centrality_positive
    .sort_values(
        "pagerank",
        ascending=False
    )
    .groupby("community")
    .first()
    .reset_index()
)

top_influencer_positive

,community,username,pagerank,in_degree,out_degree,betweenness
0,0,teeadnu,0.005618,1,1,0.0
1,1,KemenkesRI,0.001491,5,0,0.0
2,2,vincentrcrd,0.001561,1,0,0.0
3,3,_fauzanx4,0.005618,1,1,0.0
4,4,_lavraa,0.005618,1,1,0.0
...,...,...,...,...,...,...
195,195,susierna,0.005618,1,1,0.0
196,196,nadiazmn___,0.001202,1,0,0.0
197,197,BTS_KJINS,0.001023,1,0,0.0
198,198,subtanyarl,0.001561,1,0,0.0


In [ ]:
top_influencer_positive.to_excel(
    os.path.join(OUTPUT_DIR, "positive_top_influencer_per_community.xlsx"),
    index=False
)

Dominant Keywords per Community

In [219]:
positive_tweets = (
    df[
        df["sentiment"] == "positive"
    ]
)

In [220]:
community_positive_tweets = (
    positive_tweets
    .merge(
        community_positive_df,
        on="username",
        how="inner"
    )
)

community_positive_tweets.head()

,tweet_id,display_name,username,followers,tweet,created_at,lang,user_location,tweet_location,likes,...,city_clean,sentiment_roberta,confidence_roberta,polarity_roberta,sentiment_model2,confidence_model2,polarity_model2,sentiment,sentiment_score,community
0,1382274159166640128,Jussie HR Era 🏒,jusjussie219,863,@ameamakunai Adaaa. Pas itu sedang marak penya...,Wed Apr 14 10:06:55 +0000 2021,in,Grand Line,NaN,0,...,NaN,positive,0.829382,0.829382,positive,0.737090,0.737090,positive,0.737090,102
1,1378298918761497088,WIN’,taehyungluvlyy,1507,@subtanyarl pernah 2 kali dengan penyakit yang...,Sat Apr 03 10:50:44 +0000 2021,in,♡,NaN,0,...,NaN,negative,0.976586,-0.976586,positive,0.771661,0.771661,positive,0.771661,198
2,1404779915690480128,STUDI KLUB SEJARAH UI,sksui,2070,Adanya Hari Demam Berdarah Dengue (DBD) ASEAN ...,Tue Jun 15 12:36:46 +0000 2021,in,"FIB UI, Depok",NaN,0,...,depok,neutral,0.977436,0.000000,positive,0.916698,0.916698,positive,0.916698,193
3,1404676887188677120,ISMKI Wilayah 4,ismki_wilayah4,2499,Hal tersebut menjadikan Indonesia adalah salah...,Tue Jun 15 05:47:22 +0000 2021,in,Indonesia,NaN,0,...,NaN,positive,0.416689,0.416689,positive,0.887027,0.887027,positive,0.887027,94
4,1404600780632444928,Faris I Pramasubakti,farisirfanp,152,@edwardsuhadi @KawalCOVID19 Sempet nemu kasus ...,Tue Jun 15 00:44:56 +0000 2021,in,BDO - BTH - KNO,NaN,12,...,NaN,positive,0.932354,0.932354,positive,0.606864,0.606864,positive,0.606864,61


In [221]:
def get_top_keywords(texts, n=10):

    vectorizer = CountVectorizer(
        max_features=1000
    )

    X = vectorizer.fit_transform(
        texts.dropna()
    )

    words = (
        vectorizer
        .get_feature_names_out()
    )

    freq = X.sum(
        axis=0
    ).A1

    keyword_df = pd.DataFrame({
        "keyword": words,
        "frequency": freq
    })

    return (
        keyword_df
        .sort_values(
            "frequency",
            ascending=False
        )
        .head(n)
    )

In [222]:
keyword_results_positive = []

for community_id in sorted(
    community_positive_tweets[
        "community"
    ].unique()
):

    texts = (
        community_positive_tweets[
            community_positive_tweets[
                "community"
            ]
            == community_id
        ]["tweet_clean"]
    )

    top_keywords = (
        get_top_keywords(
            texts,
            n=10
        )
    )

    top_keywords[
        "community"
    ] = community_id

    keyword_results_positive.append(
        top_keywords
    )

In [223]:
community_keywords_positive = pd.concat(
    keyword_results_positive,
    ignore_index=True
)

community_keywords_positive.head()

,keyword,frequency,community
0,dbd,7,0
1,di,4,0
2,semua,4,0
3,dengue,3,0
4,lawan,3,0


In [ ]:
community_keywords_positive.to_excel(
    os.path.join(OUTPUT_DIR, "positive_community_keywords.xlsx"),
    index=False
)

------------- Neutral Network ------------------

In [ ]:
top_neutral = pd.read_excel(os.path.join(OUTPUT_DIR, "neutral_network_centrality.xlsx"))

In [226]:
neutral_edges = (
    edges_weighted[
        edges_weighted["sentiment"]
        == "neutral"
    ]
)
print(
    "Neutral edges:",
    len(neutral_edges)
)

Neutral edges: 757


In [227]:
G_neutral = nx.from_pandas_edgelist(
    neutral_edges,
    source="source",
    target="target",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

In [228]:
print(
    "Nodes:",
    G_neutral.number_of_nodes()
)

print(
    "Edges:",
    G_neutral.number_of_edges()
)

Nodes: 933
Edges: 757


Community Detection

In [229]:
partition_neutral = (
    community_louvain.best_partition(
        G_neutral.to_undirected(),
        random_state=42
    )
)

In [230]:
print(
    "Number of communities:",
    len(
        set(
            partition_neutral.values()
        )
    )
)

Number of communities: 426


In [231]:
community_neutral_df = pd.DataFrame(
    partition_neutral.items(),
    columns=[
        "username",
        "community"
    ]
)

community_neutral_df.head()

,username,community
0,0bungkal,0
1,polres_ponorogo,0
2,0jenry1,1
3,PNS_Ababil,1
4,1minggu1cerita,2


In [ ]:
community_neutral_df.to_excel(
    os.path.join(OUTPUT_DIR, "neutral_community_assignment.xlsx"),
    index=False
)

Community Size

In [233]:
community_size_neutral = (
    community_neutral_df["community"]
    .value_counts()
    .reset_index()
)

community_size_neutral.columns = [
    "community",
    "members"
]

In [234]:
community_size_neutral = (
    community_size_neutral
    .sort_values(
        "members",
        ascending=False
    )
)

community_size_neutral.head(20)

,community,members
0,199,48
1,24,24
2,23,16
3,359,16
4,309,10
5,231,10
6,104,9
7,13,8
8,217,8
9,88,7


In [ ]:
community_size_neutral.to_excel(
    os.path.join(OUTPUT_DIR, "neutral_community_size.xlsx"),
    index=False
)

Top Influencer per Community

In [236]:
community_centrality_neutral = (
    community_neutral_df
    .merge(
        top_neutral,
        on="username",
        how="left"
    )
)

In [237]:
community_centrality_neutral.head()

,username,community,pagerank,in_degree,out_degree,betweenness
0,0bungkal,0,0.000426,0,1,0.0
1,polres_ponorogo,0,0.000788,1,0,0.0
2,0jenry1,1,0.000426,0,1,0.0
3,PNS_Ababil,1,0.000788,1,0,0.0
4,1minggu1cerita,2,0.000426,0,1,0.0


In [238]:
top_influencer_neutral = (
    community_centrality_neutral
    .sort_values(
        "pagerank",
        ascending=False
    )
    .groupby("community")
    .first()
    .reset_index()
)

In [239]:
top_influencer_neutral.head()

,community,username,pagerank,in_degree,out_degree,betweenness
0,0,polres_ponorogo,0.000788,1,0,0.0
1,1,PNS_Ababil,0.000788,1,0,0.0
2,2,nuellubis,0.000788,1,0,0.0
3,3,nmonarizqa,0.000788,1,0,0.0
4,4,sisilkrispi_,0.000788,2,0,0.0


In [ ]:
top_influencer_neutral.to_excel(
    os.path.join(OUTPUT_DIR, "neutral_top_influencer_per_community.xlsx"),
    index=False
)

Top Keywords per Community

In [241]:
neutral_tweets = (
    df[
        df["sentiment"] == "neutral"
    ]
)

In [242]:
community_neutral_tweets = (
    neutral_tweets
    .merge(
        community_neutral_df,
        on="username",
        how="inner"
    )
)

In [243]:
community_sentiment_neutral = (
    community_neutral_tweets
    .groupby(
        [
            "community",
            "sentiment"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

In [244]:
dominant_sentiment_neutral = (
    community_sentiment_neutral
    .sort_values(
        "count",
        ascending=False
    )
    .groupby("community")
    .first()
    .reset_index()
)

In [ ]:
dominant_sentiment_neutral.to_excel(
    os.path.join(OUTPUT_DIR, "neutral_dominant_sentiment.xlsx"),
    index=False
)

In [246]:
def get_top_keywords(
    texts,
    n=10
):

    vectorizer = CountVectorizer(
        max_features=1000
    )

    X = vectorizer.fit_transform(
        texts.dropna()
    )

    words = (
        vectorizer
        .get_feature_names_out()
    )

    freq = X.sum(
        axis=0
    ).A1

    keyword_df = pd.DataFrame({
        "keyword": words,
        "frequency": freq
    })

    return (
        keyword_df
        .sort_values(
            "frequency",
            ascending=False
        )
        .head(n)
    )

In [247]:
keyword_results_neutral = []

for community_id in sorted(
    community_neutral_tweets[
        "community"
    ].unique()
):

    texts = (
        community_neutral_tweets[
            community_neutral_tweets[
                "community"
            ]
            == community_id
        ]["tweet_clean"]
    )

    top_keywords = (
        get_top_keywords(
            texts,
            n=10
        )
    )

    top_keywords[
        "community"
    ] = community_id

    keyword_results_neutral.append(
        top_keywords
    )

In [248]:
community_keywords_neutral = pd.concat(
    keyword_results_neutral,
    ignore_index=True
)

community_keywords_neutral.head()

,keyword,frequency,community
0,bungkal,3,0
1,puskesmas,2,0
2,anwar,1,0
3,berdarah,1,0
4,bersama,1,0


In [ ]:
community_keywords_neutral.to_excel(
    os.path.join(OUTPUT_DIR, "neutral_community_keywords.xlsx"),
    index=False
)

-------------------- Negative Network --------------------------

In [ ]:
top_negative = pd.read_excel(os.path.join(OUTPUT_DIR, "negative_network_centrality.xlsx"))

In [251]:
negative_edges = (
    edges_weighted[
        edges_weighted["sentiment"]
        == "negative"
    ]
)

In [252]:
print(
    "Negative edges:",
    len(negative_edges)
)

Negative edges: 812


In [253]:
G_negative = nx.from_pandas_edgelist(
    negative_edges,
    source="source",
    target="target",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

In [254]:
print(
    "Nodes:",
    G_negative.number_of_nodes()
)

print(
    "Edges:",
    G_negative.number_of_edges()
)

Nodes: 1081
Edges: 812


Community detection

In [255]:
partition_negative = (
    community_louvain.best_partition(
        G_negative.to_undirected(),
        random_state=42
    )
)

In [256]:
print(
    "Number of communities:",
    len(
        set(
            partition_negative.values()
        )
    )
)

Number of communities: 426


In [257]:
community_negative_df = pd.DataFrame(
    partition_negative.items(),
    columns=[
        "username",
        "community"
    ]
)

community_negative_df.head()

,username,community
0,6v66le,0
1,dibsturb,0
2,kenmeong,0
3,99cheeze,1
4,bronistay,1


In [ ]:
community_negative_df.to_excel(
    os.path.join(OUTPUT_DIR, "negative_community_assignment.xlsx"),
    index=False
)

Community Size

In [259]:
community_size_negative = (
    community_negative_df["community"]
    .value_counts()
    .reset_index()
)

community_size_negative.columns = [
    "community",
    "members"
]

In [260]:
community_size_negative = (
    community_size_negative
    .sort_values(
        "members",
        ascending=False
    )
)

community_size_negative.head(20)

,community,members
0,15,34
1,127,23
2,179,19
3,21,15
4,68,15
5,28,13
6,32,13
7,8,12
8,249,11
9,18,10


In [ ]:
community_size_negative.to_excel(
    os.path.join(OUTPUT_DIR, "negative_community_size.xlsx"),
    index=False
)

Top Influencer per Commuity

In [262]:
community_centrality_negative = (
    community_negative_df
    .merge(
        top_negative,
        on="username",
        how="left"
    )
)

community_centrality_negative.head()

,username,community,pagerank,in_degree,out_degree,betweenness
0,6v66le,0,0.000463,0,2,0.0
1,dibsturb,0,0.000659,1,0,0.0
2,kenmeong,0,0.000659,1,0,0.0
3,99cheeze,1,0.000463,0,1,0.0
4,bronistay,1,0.000856,1,0,0.0


In [263]:
top_influencer_negative = (
    community_centrality_negative
    .sort_values(
        "pagerank",
        ascending=False
    )
    .groupby("community")
    .first()
    .reset_index()
)

top_influencer_negative.head()

,community,username,pagerank,in_degree,out_degree,betweenness
0,0,kenmeong,0.000659,1,0,0.0
1,1,bronistay,0.000856,1,0,0.0
2,2,arlyanti_,0.000856,1,0,0.0
3,3,trendingtopiq,0.000856,1,0,0.0
4,4,cobeh09,0.000659,1,0,0.0


In [ ]:
top_influencer_negative.to_excel(
    os.path.join(OUTPUT_DIR, "negative_top_influencer_per_community.xlsx"),
    index=False
)

Dominant Sentiment per Community

In [265]:
negative_tweets = (
    df[
        df["sentiment"] == "negative"
    ]
)

In [266]:
community_negative_tweets = (
    negative_tweets
    .merge(
        community_negative_df,
        on="username",
        how="inner"
    )
)

community_negative_tweets.head()

,tweet_id,display_name,username,followers,tweet,created_at,lang,user_location,tweet_location,likes,...,city_clean,sentiment_roberta,confidence_roberta,polarity_roberta,sentiment_model2,confidence_model2,polarity_model2,sentiment,sentiment_score,community
0,1406970754688717056,Яizal do,afrkml,350301,Gejala yg ditimbulkan memang umum dan sudah di...,Mon Jun 21 13:42:22 +0000 2021,in,Dukung kami di sini →,NaN,245,...,NaN,negative,0.752307,-0.752307,negative,0.995110,-0.995110,negative,-0.995110,15
1,1407197145682247936,tikatiki,tikaresianaa,13,@milasantika06 Nah betul tuh\nTapi akhir akhir...,Tue Jun 22 04:41:58 +0000 2021,in,"Purwokerto Utara, Indonesia",NaN,0,...,NaN,negative,0.996815,-0.996815,negative,0.995957,-0.995957,negative,-0.995957,387
2,1404754951893443072,pin.,albusvenetus,459,"@bertanyarl 3,5,8,9,10 ini yg blm\n1. Taun lal...",Tue Jun 15 10:57:34 +0000 2021,in,She/her 20+,NaN,0,...,NaN,negative,0.736742,-0.736742,negative,0.984341,-0.984341,negative,-0.984341,23
3,1406566917880434944,Askrlfess,askrlfess,707097,[askrl] penyakit dbd sama malaria sama gak sih?,Sun Jun 20 10:57:40 +0000 2021,in,"Jakarta, Indonesia",NaN,0,...,jakarta,negative,0.719211,-0.719211,negative,0.956276,-0.956276,negative,-0.956276,38
4,1407708433021764096,🚫Zionism,dawil_k,16,"@mysubuh Kemana TBC, Kanker, DBD dan penyakit ...",Wed Jun 23 14:33:39 +0000 2021,in,-,NaN,0,...,NaN,negative,0.993563,-0.993563,negative,0.993711,-0.993711,negative,-0.993711,108


In [267]:
community_sentiment_negative = (
    community_negative_tweets
    .groupby(
        [
            "community",
            "sentiment"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

In [268]:
dominant_sentiment_negative = (
    community_sentiment_negative
    .sort_values(
        "count",
        ascending=False
    )
    .groupby("community")
    .first()
    .reset_index()
)

dominant_sentiment_negative

,community,sentiment,count
0,0,negative,1
1,1,negative,1
2,2,negative,1
3,3,negative,1
4,4,negative,1
...,...,...,...
421,421,negative,1
422,422,negative,1
423,423,negative,2
424,424,negative,1


In [ ]:
dominant_sentiment_negative.to_excel(
    os.path.join(OUTPUT_DIR, "negative_dominant_sentiment.xlsx"),
    index=False
)

Top Keywords per Community

In [270]:
def get_top_keywords(
    texts,
    n=10
):

    vectorizer = CountVectorizer(
        max_features=1000
    )

    X = vectorizer.fit_transform(
        texts.dropna()
    )

    words = (
        vectorizer
        .get_feature_names_out()
    )

    freq = X.sum(
        axis=0
    ).A1

    keyword_df = pd.DataFrame({
        "keyword": words,
        "frequency": freq
    })

    return (
        keyword_df
        .sort_values(
            "frequency",
            ascending=False
        )
        .head(n)
    )

In [271]:
keyword_results_negative = []

for community_id in sorted(
    community_negative_tweets[
        "community"
    ].unique()
):

    texts = (
        community_negative_tweets[
            community_negative_tweets[
                "community"
            ]
            == community_id
        ]["tweet_clean"]
    )

    top_keywords = (
        get_top_keywords(
            texts,
            n=10
        )
    )

    top_keywords[
        "community"
    ] = community_id

    keyword_results_negative.append(
        top_keywords
    )

In [272]:
community_keywords_negative = pd.concat(
    keyword_results_negative,
    ignore_index=True
)

community_keywords_negative.head()

,keyword,frequency,community
0,berdarah,1,0
1,demam,1,0
2,tingkyuuu,1,0
3,banget,1,1
4,biasanya,1,1


In [ ]:
community_keywords_negative.to_excel(
    os.path.join(OUTPUT_DIR, "negative_community_keywords.xlsx"),
    index=False
)